# AI Sommelier RAG: 와인 리뷰 기반 와인 추천 서비스

In [20]:
# %pip install -qU langchain-pinecone pinecone

In [21]:
from dotenv import load_dotenv
load_dotenv()

True

## pinecone vector DB 준비

In [22]:
TEXT_MODEL = 'gpt-4.1-mini'
VISION_MODEL = 'gpt-4.1-mini'
EMBEDDING_MODEL = 'text-embedding-3-small'
EMBEDDING_DIM = 1536
PINECONE_INDEX_NAME = 'winemeg-review-data'
PINEONE_CLOUD = 'aws'
PINECONE_REGION = 'us-east-1'

In [23]:
from pinecone import Pinecone, ServerlessSpec

# .env 파일에 PINECONE_API_KEY 설정이 추가되고 환경 변수가 로드 되어야 함
pc = Pinecone()

# 같은 이름의 Index가 없을 때만 새로 생성한다.
if not pc.has_index(PINECONE_INDEX_NAME):
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIM,
        metric="cosine",
        spec=ServerlessSpec(
            cloud=PINEONE_CLOUD,
            region=PINECONE_REGION
        )
    )
    print("PINECONE INDEX 생성 완료 :",PINECONE_INDEX_NAME)
else:
    print("이미 존재하는 PINECONE INDEX 사용 :",PINECONE_INDEX_NAME)

Pinecone_index = pc.Index(PINECONE_INDEX_NAME)
Pinecone_index.describe_index_stats()

이미 존재하는 PINECONE INDEX 사용 : winemeg-review-data


{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 3000}},
 'total_vector_count': 3000,
 'vector_type': 'dense'}

## 와인 리뷰 CSV를 Document로 변환

In [24]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(
    file_path="data/winemag-data-130k-v2.csv",
    encoding='utf-8'
)

all_docs = loader.load()

print("전체 document 수:",len(all_docs))

전체 document 수: 65499


In [25]:
# 전체 데이터 저장시 시간, 비용이 많이 들기 때문에 제한해서 사용한다.
MAX_DOCS = 3000

docs = all_docs if MAX_DOCS is None else all_docs[:MAX_DOCS]

print("이번 테스트에서 사용할 Document 수 :", len(docs))

for i,doc in enumerate(docs[:2],start=1):
    print(f"[Document {i}]")
    print("metadata : ",doc.metadata)
    print(doc.page_content)

이번 테스트에서 사용할 Document 수 : 3000
[Document 1]
metadata :  {'source': 'data/winemag-data-130k-v2.csv', 'row': 0}
: 0
country: Italy
description: Aromas include tropical fruit, broom, brimstone and dried herb. The palate isn't overly expressive, offering unripened apple, citrus and dried sage alongside brisk acidity.
designation: Vulkà Bianco
points: 87
price: 
province: Sicily & Sardinia
region_1: Etna
region_2: 
taster_name: Kerin O’Keefe
taster_twitter_handle: @kerinokeefe
title: Nicosia 2013 Vulkà Bianco  (Etna)
variety: White Blend
winery: Nicosia
[Document 2]
metadata :  {'source': 'data/winemag-data-130k-v2.csv', 'row': 1}
: 1
country: Portugal
description: This is ripe and fruity, a wine that is smooth while still structured. Firm tannins are filled out with juicy red berry fruits and freshened with acidity. It's  already drinkable, although it will certainly be better from 2016.
designation: Avidagos
points: 87
price: 15
province: Douro
region_1: 
region_2: 
taster_name: Roger Vos

## Pinecore Vector Store 연결

In [26]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = PineconeVectorStore(
    index=Pinecone_index,
    embedding=embeddings
)

print("PineconeVectorStore 연결 완료")

PineconeVectorStore 연결 완료


## Document를 Pinecone에 저장

In [27]:
from tqdm.auto import tqdm

BATCH_SIZE = 100

for start in tqdm(range(0,len(docs),BATCH_SIZE)):
    end = start + BATCH_SIZE
    batch = docs[start:end]

    # row 번호를 사용하면 같은 데이터를 다시 실행해도 같은 id가 만들어진다.
    ids = [
        f"winmeg-{doc.metadata.get('row',start+offset)}"
        for offset, doc in enumerate(batch)
    ]

    vector_store.add_documents(
        documents=batch,
        ids=ids
    )

print("Pinecone 저장 완료")
print(Pinecone_index.describe_index_stats())

  0%|          | 0/30 [00:00<?, ?it/s]

Pinecone 저장 완료
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 3000}},
 'total_vector_count': 3000,
 'vector_type': 'dense'}


## 검색 결과 검증
- 검색어를 직접 넣어보고 의도한 와인 리뷰가 검색 되는지 확인 (RAG 구현 전단계)

In [28]:
retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs={"k":3}
)

test_queries = [
    'full-bodied red wine with dark fruit and oak',
    'fresh white wine with citrus and high acidity',
    'sweet desert wine with honey aroma'
]

for query in test_queries:
    print("검색어 : ", query)
    result = retriever.invoke(query)

    for i,doc in enumerate(result,start=1):
        print(f"[검색결과 {i}]")
        print("metadata : ", doc.metadata)
        print(doc.page_content[:1000])

    print("-"*100)

검색어 :  full-bodied red wine with dark fruit and oak
[검색결과 1]
metadata :  {'row': 1354.0, 'source': 'data/winemag-data-130k-v2.csv'}
: 1354
country: US
description: Strong in black currant, raisin, blackberry pie and oaky flavors, this full-bodied effort has thick tannins. It's pleasant for drinking now with barbecue and roasts.
designation: 
points: 86
price: 30
province: California
region_1: California
region_2: California Other
taster_name: 
taster_twitter_handle: 
title: Dark Hundred 2011 Red (California)
variety: Red Blend
winery: Dark Hundred
[검색결과 2]
metadata :  {'row': 1771.0, 'source': 'data/winemag-data-130k-v2.csv'}
: 1771
country: US
description: A very deep red color catches the attention. Next, strong, wild aromas like smoke and black rubber lead to a rather fruity but still firm and tannic texture. The flavors are more like blackberries and blueberries, but the overall effect is slightly viscous.
designation: 
points: 84
price: 9
province: California
region_1: California


## LLM 단독 추천 확인

In [29]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    model=TEXT_MODEL
)

simple_recommand_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "당신은 친절한 소믈리에입니다. 음식과 잘 어울리는 와인을 초보자도 이해하기 쉽게 추천해주세요."
    ),
    (
        "human",
        "다음 음식에 어울리는 와인을 추천해주세요. \n\n음식 : {dish}"
    )
])

simple_recommend_chain = simple_recommand_prompt | llm | StrOutputParser()

print(simple_recommend_chain.invoke({"dish":"로즈마리를 곁들인 스테이크"}))

로즈마리를 곁들인 스테이크는 풍부하고 진한 맛이 특징이라, 그 맛을 잘 살려줄 와인을 고르는 게 중요해요.

초보자분들께 추천드리는 와인은 **카베르네 소비뇽(Cabernet Sauvignon)**입니다. 이 와인은 진한 과일 향과 탄닌이 스테이크의 육즙과 잘 어울리고, 로즈마리의 허브 향과도 조화를 이룹니다.

만약 조금 더 부드러운 맛을 원하시면, **말벡(Malbec)**이나 **쉬라즈(Shiraz)**도 좋은 선택이에요. 이 와인들은 과일 풍미가 풍부하면서도 스테이크와 잘 맞는 스파이스 향이 있어요.

요약하자면:
- 진한 맛과 탄닌이 좋아: 카베르네 소비뇽
- 부드럽고 풍부한 과일향 좋아: 말벡, 쉬라즈

맛있게 즐기시길 바랄게요!


## 음식 설명으로 와인 리뷰 검색

In [30]:
sample_dish_flavor = (
    "A juicy grilled steak with rosemary aroma, roasted vegetables"
    "savory meat flavor, and a rich smoky finish"
)

sample_docs = retriever.invoke(sample_dish_flavor)

for i,doc in enumerate(sample_docs,start=1):
    print(f"[검색결과 {i}]")
    print("metadata : ", doc.metadata)
    print(doc.page_content[:1000])
    print("-"*100)

[검색결과 1]
metadata :  {'row': 264.0, 'source': 'data/winemag-data-130k-v2.csv'}
: 264
country: South Africa
description: A good amount of earthy spice, tea leaves and forrest floor lead the way on the nose, but the black cherry and berry fruit aromas follow shortly after with an additional accent of sweet cured meat. Medium weight and lush, the creamy mouth transitions into a finish loaded with sweet spice and bittersweet cocoa.
designation: 
points: 89
price: 19
province: Stellenbosch
region_1: 
region_2: 
taster_name: Lauren Buzzeo
taster_twitter_handle: @laurbuzz
title: Jardin 2007 Syrah (Stellenbosch)
variety: Syrah
winery: Jardin
----------------------------------------------------------------------------------------------------
[검색결과 2]
metadata :  {'row': 1374.0, 'source': 'data/winemag-data-130k-v2.csv'}
: 1374
country: US
description: Made in a soft, gentle way, this pretty Mourvèdre has chocolate-infused blackberry, currant, raspberry, licorice, cola and pepper flavors. It's v

In [31]:
def format_wine_docs(docs : list) -> str:
    """검색된 와인 리뷰 Document를 추천 prompt에 넣기 좋은 문자열로 변환하는 함수"""

    formatted = []

    for i,doc in enumerate(docs,start=1):
        source = doc.metadata.get("source","unknown")
        row = doc.metadata.get("row","unknown")

        formatted.append(
            f"[와인리뷰 {i}]\n"
            f"source : {source}\n"
            f"row : {row}\n"
            f"content : \n{doc.page_content}"
            )
    
    return "\n\n".join(formatted)

## 음식 이미지를 풍미 설명으로 변환

In [32]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.runnables import RunnableLambda

vision_llm = ChatOpenAI(
    model=VISION_MODEL
)

def describe_dish_flavor(query: dict) -> str:
    """음식 이미지 URL 목록을 받아 와인 페어링에 필요한 풍미 설명을 문자열로 반환한다."""

    image_urls = query.get("image_urls", [])

    if not image_urls:
        raise ValueError("image_urls 값이 비어 있습니다.")

    content = [
        {
            "type": "text",
            "text": (
                "Look at the food image(s) and describe the dish for wine pairing. "
                "Focus on ingredients, cooking method, sauce, texture, intensity, acidity, fat, sweetness, "
                "spiciness, and overall flavor profile. "
                "Write the answer in English because the wine review data is in English."
                "Keep it concise in 2-3 sentences and include searchable flavor keywords."
            ),
        }
    ]

    # 여러 장의 이미지를 받을 수 있도록 URL 목록을 반복 처리한다.
    for image_url in image_urls:
        content.append(
            {
                "type": "image_url",
                "image_url": {"url": image_url},
            }
        )

    messages = [
        SystemMessage(
            content=(
                "You are a culinary expert who writes concise and useful flavor descriptions "
                "for wine pairing search queries."
            )
        ),
        HumanMessage(content=content),
    ]

    response = vision_llm.invoke(messages)
    return response.content


describe_dish_flavor_chain = RunnableLambda(describe_dish_flavor)

dish_flavor = describe_dish_flavor_chain.invoke({
    "image_urls": [
        "https://justcook.butcherbox.com/wp-content/uploads/2025/02/Rib-Eye-Steak-au-Poivre-with-Roasted-Veggies--500x500.jpg"
    ]
})

print(dish_flavor)

This dish features a juicy, medium-rare grilled steak seasoned with cracked black pepper, offering a rich, savory, and slightly smoky flavor. Accompanied by charred, tender roasted vegetables including zucchini, red bell peppers, and onions, it balances fat and acidity with mild sweetness and a touch of bitterness. Key flavors: grilled steak, cracked pepper, roasted vegetables, smoky, savory, tender, balanced acidity.


## VISION MODEL이 반환한 설명으로 관련 와인 리뷰 검색

In [33]:
def search_wines(dish_flavor: str) -> dict:
    """음식 풍미 설명과 유사한 와인 리뷰를 Pinecone에서 검색한다."""

    docs = retriever.invoke(dish_flavor)

    return {
        "dish_flavor" : dish_flavor,
        "wine_reviews" : format_wine_docs(docs),
        "retrieved_docs" : docs
    }

wine_review_retrieval_chain = RunnableLambda(search_wines)
retrieval_result = wine_review_retrieval_chain.invoke(dish_flavor)

print("[음식설명]")
print(retrieval_result["dish_flavor"])

print("\n[검색 된 와인 리뷰]")
print(retrieval_result["wine_reviews"])

[음식설명]
This dish features a juicy, medium-rare grilled steak seasoned with cracked black pepper, offering a rich, savory, and slightly smoky flavor. Accompanied by charred, tender roasted vegetables including zucchini, red bell peppers, and onions, it balances fat and acidity with mild sweetness and a touch of bitterness. Key flavors: grilled steak, cracked pepper, roasted vegetables, smoky, savory, tender, balanced acidity.

[검색 된 와인 리뷰]
[와인리뷰 1]
source : data/winemag-data-130k-v2.csv
row : 1464.0
content : 
: 1464
country: US
description: From a warm-climate zone in a cool vintage, this wine is fascinatingly savory. Black and white pepper, game meat, crushed graphite and espresso aromas promise an awesome sip. It delivers with black peppercorn, green olive, black plum skin and elderberry flavors.
designation: 
points: 93
price: 38
province: California
region_1: Arroyo Seco
region_2: Central Coast
taster_name: Matt Kettmann
taster_twitter_handle: @mattkettmann
title: Mesa Del Sol 2011

## 검색된 리뷰를 근거로 와인 추천 생성

In [34]:
recommend_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a knowledgeable and friendly sommelier.

Your task is to recommend wines that pair well with the given dish.
Use the retrieved wine reviews as the main evidence.
Do not invent specific wines that are not supported by the retrieved reviews.
If the retrieved reviews are insufficient, say that the evidence is limited.

Respond in Korean.
"""
    ),
    (
        "human",
        """
[Dish flavor description]
{dish_flavor}

[Retrieved wine reviews]
{wine_reviews}

[Output format]
1. 추천 와인 스타일:
2. 추천 이유:
3. 근거로 사용한 리뷰 요약:
4. 주의할 점:
"""
    ),
])

recommend_llm = ChatOpenAI(
    model=TEXT_MODEL
)

recommend_wines_chain = recommend_prompt | recommend_llm | StrOutputParser()

recommendation = recommend_wines_chain.invoke({
    "dish_flavor": retrieval_result["dish_flavor"],
    "wine_reviews": retrieval_result["wine_reviews"],
})

print(recommendation)

1. 추천 와인 스타일:
   - 시라(Syrah)
   - 피노 누아(Pinot Noir)

2. 추천 이유:
   medium-rare로 구운 스테이크의 풍부하고 짭짤하며 약간 스모키한 맛에 잘 어울리는 흑후추를 강조한 시라 와인이 특히 적합합니다. 시라의 검은 후추와 스파이시한 향, 그리고 게임 육류와 검은 자두 등의 풍미가 스테이크의 풍미와 잘 조화를 이룹니다. 또한, 로스트 야채의 약간 쌉싸름하고 단맛이 조화된 맛을 피노 누아의 신선한 산도와 허브, 붉은 과일 향이 상쇄하면서 균형을 맞춰줍니다.

3. 근거로 사용한 리뷰 요약:
   - Mesa Del Sol 2011 Syrah (Arroyo Seco) (점수 93): 검은색과 흰 후추, 게임육, 에스프레소 향이 조화를 이루며 검은 후추 알갱이와 검은 자두, 엘더베리 풍미가 뛰어남.
   - Greenwood Ridge 2013 Estate Bottled Syrah (Mendocino Ridge) (점수 90): 독특한 흑후추 캐릭터와 다크하고 스모키한 외관, 라즈베리 노트가 살아있어 후추와 잘 어울림.
   - Presqu'ile 2012 Steiner Creek Vineyard Pinot Noir (San Luis Obispo County) (점수 92): 산뜻하고 생기있는 붉은 과일과 허브, 후추향과 달콤한 야채풍미가 균형을 이루며 우마미가 발달할 것으로 기대됨.

4. 주의할 점:
   - 스테이크의 진한 맛과 스모키함을 고려할 때, 너무 가벼운 레드 와인은 맛이 묻힐 수 있습니다.
   - 너무 높은 타닌감은 고기의 풍미를 덮을 수 있으므로, 중간 정도 바디와 산도를 가진 와인이 적합합니다.
   - 제시된 와인 중 가격대와 구입 가능성을 고려해 선택하면 좋겠습니다.


## 전체 RAG Chain 연결
- 이미지 URL을 음식 풍미 설명으로 변환한다.
- 음식 풍미 설명으로 Pinecone에서 관련 와인 리뷰를 검색한다.
- 검색된 리뷰를 근거로 와인을 추천한다.

In [35]:
ai_sommelier_rag_chain = (
    describe_dish_flavor_chain
    | wine_review_retrieval_chain
    | recommend_wines_chain
)

output = ai_sommelier_rag_chain.invoke({
    "image_urls":[
        "https://justcook.butcherbox.com/wp-content/uploads/2025/02/Rib-Eye-Steak-au-Poivre-with-Roasted-Veggies--500x500.jpg"
    ]
})

print(output)

1. 추천 와인 스타일: 신선한 산도와 블랙 페퍼 향이 돋보이는 시라(Syrah) 또는 피노 누아(Pinot Noir)

2. 추천 이유: 스테이크의 강렬한 페퍼 크러스트와 훈연된 맛이 풍부한 시라 와인의 블랙 페퍼 캐릭터와 아주 잘 어울립니다. 또한, 구운 채소의 달콤하고 흙내 나는 느낌은 피노 누아 와인의 신선한 과일 향과 약간의 허브, 향신료 노트와 조화를 이루어 균형감을 선사합니다. 특히 중간 정도 바디감과 적절한 산미가 있어 육즙 많고 진한 스테이크 맛을 돋보이게 하면서 저녁 식사의 풍미를 한층 끌어올릴 수 있습니다.

3. 근거로 사용한 리뷰 요약: 
- Greenwood Ridge 2013 Estate Bottled Syrah (Mendocino Ridge)는 중간 바디에 다크하고 훈연된 외관과 생생한 라즈베리 노트, 독특한 블랙 페퍼 특징을 가진다고 평가되었습니다. 
- Presqu'ile 2012 Steiner Creek Vineyard Pinot Noir (San Luis Obispo County)는 다양한 색상의 페퍼콘 향, 시원한 신선도와 로즈, 산딸기 힌트를 갖고 있으면서 채소와 우마미가 어우러지는 복합미를 보여줍니다. 
- Chateau Ste. Michelle 2006 Syrah (Columbia Valley)는 블랙 페퍼와 산뜻한 텍스처가 있으나 가볍고 소박한 스타일로, 피자 같은 간단한 요리와 어울리는 편입니다.

4. 주의할 점: 
- 가격과 품질 차이에 따라 와인의 복합성과 깊이가 달라지므로, 보다 진한 육류 요리와 매칭할 때는 중간 이상 가격대의 시라를 선택하는 것이 좋습니다.
- 피노 누아는 해안가 냉기 기후 산지의 와인이 잘 맞으며, 너무 강한 바디감보다는 적절한 산도와 향신료가 조화된 제품을 선택하세요.

요약하자면, 페퍼 스테이크와 구운 채소의 풍미를 최대한 살리려면 캘리포니아 멘도시노 지역의 Greenwood Ridge 2013 Estate Bottled Syrah 같은 중간 바디의 블랙 페퍼 풍미가 확실한 시라 와

## 다른 이미지로 테스트

In [37]:
test_images_urls = [
    "https://www.allrecipes.com/thmb/R4reoZMOwUpDh9SJnc5xgRYH4No=/1500x0/filters:no_upscale():max_bytes(150000):strip_icc()/221142-new-york-style-cheesecake-VAT-Beauty-2x1-4d4d4be1f12b4473ae748387fff54290.jpg",
    "https://recipe1.ezmember.co.kr/cache/recipe/2023/01/10/4a3651b8ba731f21bd63f5c5cac3ccc51.jpg",
    "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQkCTc5tiHe9cjKf-UOlWCoIG04EaCUJh-kZQ&s"
]

test_output = ai_sommelier_rag_chain.invoke({
    "image_urls":test_images_urls
})

print(test_output)

1. 추천 와인 스타일: 부드럽고 크리미한 샤르도네(Chardonnay)
2. 추천 이유: 클래식 베이크드 치즈케이크의 풍부하고 크리미한 질감, 부드러운 바닐라와 버터리한 맛을 잘 보완하며 산도가 적당히 낮아 치즈케이크의 크리미함을 해치지 않습니다.  
3. 근거로 사용한 리뷰 요약: Nica 2012 Chardonnay(캘리포니아)는 바닐라와 무화과 등의 풍부한 맛과 부드러운 질감, 그리고 낮은 산도를 가지고 있어 크림처럼 부드러운 와인입니다. 치즈케이크와 궁합이 좋습니다.  
4. 주의할 점: 너무 강한 오크향이나 과도한 산도를 가진 와인은 치즈케이크의 풍미를 압도할 수 있으니 피하는 것이 좋습니다.

---

1. 추천 와인 스타일: 상큼한 로제 와인(Rosé)
2. 추천 이유: 매콤하고 감칠맛 강한 떡볶이의 진한 양념과 향신료 맛을 깔끔하게 씻어주며, 달콤함과 매운맛을 균형 있게 중화시켜 줍니다.
3. 근거로 사용한 리뷰 요약: Cupcake 2016 Rosé(캘리포니아)는 신선한 감귤류 맛과 산도가 조화롭게 어우러져 해산물이나 짭짤한 고기류와 잘 어울립니다. 떡볶이의 매콤달콤한 맛과 조화가 기대됩니다.
4. 주의할 점: 너무 무겁거나 탄닌이 강한 레드 와인은 매운맛과 충돌할 수 있으니 가벼운 로제를 선택하세요.

---

1. 추천 와인 스타일: 리뷰 근거가 부족합니다.
2. 추천 이유: 해산물과 매운 고추장 육수의 복합적인 감칠맛, 신선한 채소, 해산물의 풍미가 조합된 매운 해물 전골에 딱 맞는 와인에 관한 리뷰가 없습니다.
3. 근거로 사용한 리뷰 요약: 해당 메뉴에 맞는 와인에 대한 직접적인 언급이 없으며, 해산물과 매운 향신료가 한꺼번에 어우러진 복합적 맛을 커버할 수 있는 와인 정보가 제한적입니다.
4. 주의할 점: 해산물의 신선함을 살리고 매운맛을 조화롭게 중화시키는 화이트 와인(예: 소비뇽 블랑)이나 발랄한 로제가 일반적으로 좋은 선택이나, 이번 리뷰 데이터에서는 관련 정보를 찾기 어려웠습니다.
